# Diamond Price Prediction ML Web App

## Overview

This project uses Linear Regression to predict diamond prices based on their characteristics, such as carat, cut, color, clarity, depth, table, and dimensions. It includes a FastAPI backend and an HTML frontend that allows users to enter diamond details and receive a predicted price.

## Requirements

Install the required libraries by running:

```bash
pip install fastapi uvicorn joblib scikit-learn pandas numpy matplotlib
```

## How to Run

1. Open `diamond_price.ipynb` in Google Colab or Jupyter Notebook.
2. Upload the `diamonds.csv` dataset. If you have `diamonds_csv.zip`, extract it first.
3. Run the notebook cells in order to explore and clean the data, train the Linear Regression model, and evaluate its performance.
4. Run the remaining cells to save the trained model and performance metrics, set up the FastAPI backend, and create the HTML frontend.
5. Run the server cell in the notebook.
6. Open the displayed web app link to start predicting diamond prices.

If you are running the app locally, start the backend with:

```bash
uvicorn main:app --reload
```

Then open `http://127.0.0.1:8000` in your browser.

## Features

- Predicts diamond prices in US dollars
- Uses Linear Regression for price prediction
- Takes diamond characteristics as input
- Estimates dimensions when exact measurements are unknown
- Displays prediction results through an HTML interface
- Shows model performance metrics, including R², MAE, and RMSE
- Keeps a history of recent predictions

## Note

The notebook expects the dataset at `/content/diamonds.csv` in Google Colab. Make sure the file is available at the correct path before running the cells.

The trained model and metrics are saved as `diamond_linear_regression.joblib` and `diamond_metrics.joblib`. Keep these files available when running the backend.

In [8]:
!pip install -q fastapi uvicorn joblib scikit-learn pandas matplotlib

## Step 1: Load the data
Run this cell and **upload `diamonds_csv.zip`** when asked (or just place it in the Colab files panel first).

In [9]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
df = pd.read_csv(r'/content/diamonds.csv')
print("Shape:", df.shape)
df.head()

Shape: (53940, 11)


,Unnamed: 0,carat,cut,color,clarity,depth,table,price,x,y,z
0,1,0.23,Ideal,E,SI2,61.5,55.0,326,3.95,3.98,2.43
1,2,0.21,Premium,E,SI1,59.8,61.0,326,3.89,3.84,2.31
2,3,0.23,Good,E,VS1,56.9,65.0,327,4.05,4.07,2.31
3,4,0.29,Premium,I,VS2,62.4,58.0,334,4.20,4.23,2.63
4,5,0.31,Good,J,SI2,63.3,58.0,335,4.34,4.35,2.75


## Step 2: Explore & clean the data
Cleaning steps we apply:
- Drop the useless index column
- Check for missing values
- Remove duplicate rows
- Remove impossible rows (a diamond with size 0 mm)
- Remove extreme outliers (typos like a 58.9 mm width)

In [10]:
# 1. Drop the unnamed index column
df = df.loc[:, ~df.columns.str.startswith("Unnamed")]

print("Missing values per column:")
print(df.isnull().sum())
print("\nDuplicate rows:", df.duplicated().sum())
print("\nRows with x, y or z = 0:", ((df.x == 0) | (df.y == 0) | (df.z == 0)).sum())


Missing values per column:
carat      0
cut        0
color      0
clarity    0
depth      0
table      0
price      0
x          0
y          0
z          0
dtype: int64

Duplicate rows: 146

Rows with x, y or z = 0: 20


In [11]:
# 1. Remove missing values
df = df.dropna()

# 2. Remove duplicate rows
df = df.drop_duplicates()

# 3. Clip extreme values
df["x"] = df["x"].clip(0.1, 10)
df["y"] = df["y"].clip(0.1, 20)
df["z"] = df["z"].clip(0.1, 10)

df["depth"] = df["depth"].clip(50, 75)
df["table"] = df["table"].clip(50, 80)

# 4. Check missing values
print("Missing values:")
print(df.isnull().sum())

# 5. Check duplicate rows
print("Duplicate rows:", df.duplicated().sum())

# 6. Check summary statistics
print(df.describe())

# 7. Check remaining rows
print("Total rows:", len(df))



Missing values:
carat      0
cut        0
color      0
clarity    0
depth      0
table      0
price      0
x          0
y          0
z          0
dtype: int64
Duplicate rows: 0
             carat         depth         table         price             x  \
count  53794.00000  53794.000000  53794.000000  53794.000000  53794.000000   
mean       0.79778     61.748318     57.458109   3933.065082      5.731206   
std        0.47339      1.424562      2.228630   3988.114460      1.120543   
min        0.20000     50.000000     50.000000    326.000000      0.100000   
25%        0.40000     61.000000     56.000000    951.000000      4.710000   
50%        0.70000     61.800000     57.000000   2401.000000      5.700000   
75%        1.04000     62.500000     59.000000   5326.750000      6.540000   
max        5.01000     75.000000     80.000000  18823.000000     10.000000   

                  y             z  
count  53794.000000  53794.000000  
mean       5.733722      3.538344  
std        1

## Step 3: Preprocessing + Linear Regression
Linear regression needs numbers, and price vs carat is curved. So we:
- **Log-transform** `carat` and the target `price` → makes the relationship a straight line
- **One-hot encode** the text columns `cut`, `color`, `clarity`

Everything is wrapped in a single **Pipeline**, so the saved model accepts raw input (the same thing the website sends).

In [12]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Select input and output
X = df[["carat", "depth", "table", "x", "y", "z",
        "cut", "color", "clarity"]]

y = np.log(df["price"])

# Convert categorical columns into numbers
encoder = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"),
     ["cut", "color", "clarity"])
], remainder="passthrough")

# Create model
model = Pipeline([
    ("encoder", encoder),
    ("regression", LinearRegression())
])

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Train the model
model.fit(X_train, y_train)

print("Model trained successfully!")
print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

Model trained successfully!
Training rows: 43035
Testing rows: 10759


## Step 4: Evaluate — R² and accuracy
For regression, "accuracy" is usually **R²** shown as a percentage. We also show how often the prediction is within ±10% of the real price.

In [13]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

# Predict prices
pred = np.exp(model.predict(X_test))
actual = np.exp(y_test)

# Calculate model performance
r2 = r2_score(actual, pred)
mae = mean_absolute_error(actual, pred)
rmse = np.sqrt(mean_squared_error(actual, pred))



print("R2 Score:", round(r2, 4))
print("R2 Percentage:", round(r2 * 100, 2), "%")
print("MAE:", round(mae, 2))
print("RMSE:", round(rmse, 2))



R2 Score: 0.9375
R2 Percentage: 93.75 %
MAE: 453.88
RMSE: 975.73


In [14]:
import joblib
import pandas as pd

# Save the trained model
joblib.dump(model, "diamond_model.joblib")

# Save model performance
metrics = {
    "R2 Score": r2,
    "MAE": mae,
    "RMSE": rmse
}

joblib.dump(metrics, "diamond_metrics.joblib")

print("Model saved successfully!")
print(metrics)

# Load the saved model
saved_model = joblib.load("diamond_model.joblib")

# Create a sample diamond
diamond = pd.DataFrame([{
    "carat": 1.0,
    "depth": 61.8,
    "table": 57,
    "x": 6.5,
    "y": 6.5,
    "z": 4.0,
    "cut": "Ideal",
    "color": "G",
    "clarity": "VS1"
}])

# Predict the diamond price
price = float(np.exp(saved_model.predict(diamond)[0]))   # model predicts log(price) -> convert back to dollars

print("Predicted diamond price: $", round(price, 2))

Model saved successfully!
{'R2 Score': 0.9375386988342711, 'MAE': 453.8792362876442, 'RMSE': np.float64(975.7251346135504)}
Predicted diamond price: $ 6279.4


## Step 6: FastAPI backend


In [16]:
import numpy as np
import os, joblib
import pandas as pd


for old in ["diamond_linear_regression.joblib", "diamond_metrics.joblib"]:
    if os.path.exists(old):
        os.remove(old)
        print("Removed old file:", old)

metrics = {
    "r2": round(float(r2), 4),
    "accuracy_percent": round(float(r2 * 100), 2),
    "mae_usd": round(float(mae), 2),
    "rmse_usd": round(float(rmse), 2),

}

joblib.dump(model, "diamond_linear_regression.joblib")
joblib.dump(metrics, "diamond_metrics.joblib")
print("Saved: diamond_linear_regression.joblib and diamond_metrics.joblib")
print(metrics)

# ---- Verify the saved model accepts RAW columns (what the website sends) ----
check = joblib.load("diamond_linear_regression.joblib")
# Update check reference step names if they match your pipeline
cols = list(check.named_steps["encoder"].feature_names_in_)
print("\nModel expects columns:", cols)
assert cols == ["carat", "depth", "table", "x", "y", "z", "cut", "color", "clarity"], \
    "Saved model has the wrong columns - re-run Steps 3, 4 and 5 in order!"

test_row = pd.DataFrame([{"carat": 1.0, "cut": "Ideal", "color": "G", "clarity": "VS1",
                          "depth": 61.8, "table": 57, "x": 6.5, "y": 6.5, "z": 4.0}])
print("Test prediction for a 1ct Ideal/G/VS1 diamond: $", round(float(np.exp(check.predict(test_row)[0])), 2))

Saved: diamond_linear_regression.joblib and diamond_metrics.joblib
{'r2': 0.9375, 'accuracy_percent': 93.75, 'mae_usd': 453.88, 'rmse_usd': 975.73}

Model expects columns: ['carat', 'depth', 'table', 'x', 'y', 'z', 'cut', 'color', 'clarity']
Test prediction for a 1ct Ideal/G/VS1 diamond: $ 6279.4


In [17]:
import numpy as np, pandas as pd, joblib
from typing import Literal
from fastapi import FastAPI, HTTPException
from fastapi.responses import HTMLResponse
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel, Field

# Load the saved model + metrics (written by the previous cell)
api_model   = joblib.load("diamond_linear_regression.joblib")
api_metrics = joblib.load("diamond_metrics.joblib")

app = FastAPI(title="Diamond Price API")
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])


class DiamondInput(BaseModel):
    carat:   float = Field(..., gt=0, le=6)
    cut:     Literal["Fair", "Good", "Very Good", "Premium", "Ideal"]
    color:   Literal["D", "E", "F", "G", "H", "I", "J"]
    clarity: Literal["I1", "SI2", "SI1", "VS2", "VS1", "VVS2", "VVS1", "IF"]
    depth:   float = Field(..., ge=40, le=80)
    table:   float = Field(..., ge=40, le=100)
    x:       float = Field(..., gt=0, le=15)
    y:       float = Field(..., gt=0, le=15)
    z:       float = Field(..., gt=0, le=10)


@app.get("/health")
def health():
    return {"status": "ok"}


@app.post("/predict/regression")
def predict_regression(item: DiamondInput):
    try:
        data = item.model_dump() if hasattr(item, "model_dump") else item.dict()
        row = pd.DataFrame([data])
        # The model was trained on log(price), so convert back to dollars with exp
        price = float(np.exp(api_model.predict(row)[0]))
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"{type(e).__name__}: {e}")
    return {
        "model": "Linear Regression",
        "predicted_price_usd": round(price, 2),
        "r2": api_metrics["r2"],
        "accuracy_percent": api_metrics["accuracy_percent"],
        "typical_error_usd": api_metrics["mae_usd"],
    }

print("API ready ✅")


API ready ✅


## Step 7: Interactive HTML frontend
The page is stored in a Python string and served by FastAPI at `/`. It has:
- a form (slider + dropdowns) that sends JSON to `/predict/regression` using `fetch`
- an animated result card
- a history of your last predictions
- an **auto-estimate** switch for x, y, z (most people don't know a diamond's millimeter size)

In [18]:
HTML_PAGE = r'''<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Diamond Price Predictor</title>
<style>
  :root { --bg1:#0f172a; --bg2:#1e3a5f; --card:#ffffff; --ink:#0f172a; --muted:#64748b; --accent:#2563eb; --line:#e2e8f0; }
  * { box-sizing: border-box; }
  body { margin:0; font-family: "Segoe UI", Roboto, Arial, sans-serif; color:var(--ink);
         background: linear-gradient(135deg, var(--bg1), var(--bg2)); min-height:100vh; padding:24px; }
  .wrap { max-width: 960px; margin: 0 auto; }
  h1 { color:#fff; margin:0 0 4px; font-size: 28px; }
  .sub { color:#cbd5e1; margin:0 0 20px; }
  .grid { display:grid; grid-template-columns: 1.3fr 1fr; gap:18px; }
  @media (max-width: 760px) { .grid { grid-template-columns: 1fr; } }
  .card { background:var(--card); border-radius:16px; padding:20px; box-shadow:0 10px 30px rgba(0,0,0,.25); }
  label { display:block; font-size:13px; font-weight:600; margin:12px 0 4px; }
  .hint { font-weight:400; color:var(--muted); }
  input[type=number], select { width:100%; padding:10px; border:1px solid var(--line); border-radius:10px; font-size:15px; background:#fff; }
  input[type=range] { width:100%; accent-color: var(--accent); }
  .row { display:grid; grid-template-columns: 1fr 1fr; gap:12px; }
  .row3 { display:grid; grid-template-columns: 1fr 1fr 1fr; gap:10px; }
  .carat-val { float:right; font-weight:700; color:var(--accent); }
  .check { display:flex; align-items:center; gap:8px; margin-top:14px; font-size:13px; color:var(--muted); }
  .btns { display:flex; gap:10px; margin-top:18px; }
  button { border:0; border-radius:10px; padding:12px 16px; font-size:15px; font-weight:600; cursor:pointer; }
  .primary { background:var(--accent); color:#fff; flex:1; }
  .primary:hover { background:#1d4ed8; }
  .ghost { background:#eef2ff; color:#3730a3; }
  .ghost:hover { background:#e0e7ff; }
  button:disabled { opacity:.6; cursor:wait; }
  .result { text-align:center; padding: 14px 0 6px; }
  .price { font-size:44px; font-weight:800; color:var(--accent); margin:6px 0; }
  .placeholder { color:var(--muted); }
  .pill { display:inline-block; background:#ecfdf5; color:#047857; border-radius:999px; padding:4px 12px; font-size:12px; font-weight:600; margin:3px; }
  .err { background:#fef2f2; color:#b91c1c; border-radius:10px; padding:10px; margin-top:12px; font-size:14px; display:none; }
  .spinner { display:none; width:26px; height:26px; margin:10px auto; border:3px solid #cbd5e1; border-top-color:var(--accent); border-radius:50%; animation: spin 0.8s linear infinite; }
  @keyframes spin { to { transform: rotate(360deg); } }
  h3 { margin:18px 0 6px; font-size:14px; color:var(--muted); text-transform:uppercase; letter-spacing:.05em; }
  table { width:100%; border-collapse:collapse; font-size:13px; }
  td, th { text-align:left; padding:6px 4px; border-bottom:1px solid var(--line); }
  th { color:var(--muted); font-weight:600; }
</style>
</head>
<body>
<div class="wrap">
  <h1>💎 Diamond Price Predictor</h1>
  <p class="sub">Linear Regression model served by FastAPI &middot; <code>POST /predict/regression</code></p>

  <div class="grid">
    <div class="card">
      <label>Carat <span class="carat-val" id="caratLabel">1.00</span></label>
      <input type="range" id="caratRange" min="0.2" max="5" step="0.01" value="1">

      <div class="row">
        <div>
          <label>Cut <span class="hint">(quality)</span></label>
          <select id="cut">
            <option>Fair</option><option>Good</option><option>Very Good</option><option>Premium</option><option selected>Ideal</option>
          </select>
        </div>
        <div>
          <label>Color <span class="hint">(D best, J worst)</span></label>
          <select id="color">
            <option>D</option><option>E</option><option>F</option><option selected>G</option><option>H</option><option>I</option><option>J</option>
          </select>
        </div>
      </div>

      <label>Clarity <span class="hint">(I1 worst, IF best)</span></label>
      <select id="clarity">
        <option>I1</option><option>SI2</option><option>SI1</option><option>VS2</option><option selected>VS1</option><option>VVS2</option><option>VVS1</option><option>IF</option>
      </select>

      <div class="row">
        <div><label>Depth % <span class="hint">(43-79)</span></label><input type="number" id="depth" value="61.8" step="0.1"></div>
        <div><label>Table % <span class="hint">(43-95)</span></label><input type="number" id="table" value="57" step="0.1"></div>
      </div>

      <div class="check"><input type="checkbox" id="auto" checked><label for="auto" style="margin:0;font-weight:400">Auto-estimate x, y, z (mm) from carat</label></div>
      <div class="row3">
        <div><label>x (mm)</label><input type="number" id="x" step="0.01" disabled></div>
        <div><label>y (mm)</label><input type="number" id="y" step="0.01" disabled></div>
        <div><label>z (mm)</label><input type="number" id="z" step="0.01" disabled></div>
      </div>

      <div class="btns">
        <button class="primary" id="predictBtn">Predict price</button>
        <button class="ghost" id="exampleBtn" type="button">Random example</button>
      </div>
      <div class="err" id="err"></div>
    </div>

    <div>
      <div class="card result">
        <div class="placeholder" id="placeholder">Fill the form and press <b>Predict price</b></div>
        <div class="spinner" id="spinner"></div>
        <div id="resultBox" style="display:none">
          <div class="placeholder">Estimated price</div>
          <div class="price" id="price">$0</div>
          <div id="pills"></div>
        </div>
      </div>
      <div class="card" style="margin-top:18px">
        <h3 style="margin-top:0">Recent predictions</h3>
        <table><thead><tr><th>Carat</th><th>Cut</th><th>Color</th><th>Clarity</th><th>Price</th></tr></thead>
        <tbody id="history"><tr><td colspan="5" class="placeholder">Nothing yet</td></tr></tbody></table>
      </div>
    </div>
  </div>
</div>

<script>
const $ = (id) => document.getElementById(id);
const history = [];

// Rough size of a round diamond from its carat weight
function estimateDims() {
  const c = parseFloat($("caratRange").value);
  const d = 6.5 * Math.cbrt(c);
  $("x").value = d.toFixed(2);
  $("y").value = d.toFixed(2);
  $("z").value = (d * 0.62).toFixed(2);
}

function syncAuto() {
  const on = $("auto").checked;
  ["x", "y", "z"].forEach(id => $(id).disabled = on);
  if (on) estimateDims();
}

$("caratRange").addEventListener("input", () => {
  $("caratLabel").textContent = parseFloat($("caratRange").value).toFixed(2);
  if ($("auto").checked) estimateDims();
});
$("auto").addEventListener("change", syncAuto);

// Count-up animation for the price
function animatePrice(target) {
  const start = performance.now(), dur = 700;
  function tick(now) {
    const t = Math.min((now - start) / dur, 1);
    const eased = 1 - Math.pow(1 - t, 3);
    $("price").textContent = "$" + Math.round(target * eased).toLocaleString();
    if (t < 1) requestAnimationFrame(tick);
  }
  requestAnimationFrame(tick);
}

function showError(msg) { $("err").textContent = msg; $("err").style.display = "block"; }

function renderHistory() {
  $("history").innerHTML = history.slice(0, 5).map(h =>
    "<tr><td>" + h.carat + "</td><td>" + h.cut + "</td><td>" + h.color + "</td><td>" + h.clarity +
    "</td><td><b>$" + Math.round(h.price).toLocaleString() + "</b></td></tr>").join("");
}

async function predict() {
  $("err").style.display = "none";
  const payload = {
    carat: parseFloat($("caratRange").value),
    cut: $("cut").value,
    color: $("color").value,
    clarity: $("clarity").value,
    depth: parseFloat($("depth").value),
    table: parseFloat($("table").value),
    x: parseFloat($("x").value),
    y: parseFloat($("y").value),
    z: parseFloat($("z").value),
  };
  if (Object.values(payload).some(v => v === "" || Number.isNaN(v))) {
    showError("Please fill in every field with a valid number.");
    return;
  }

  $("predictBtn").disabled = true;
  $("placeholder").style.display = "none";
  $("resultBox").style.display = "none";
  $("spinner").style.display = "block";

  try {
    // relative URL, so it works inside Colab's proxy and on localhost
    const res = await fetch("predict/regression", {
      method: "POST",
      headers: { "Content-Type": "application/json" },
      body: JSON.stringify(payload),
    });
    const raw = await res.text();
    let data;
    try { data = JSON.parse(raw); }
    catch { data = { detail: "Server replied: " + raw.slice(0, 200) }; }
    if (!res.ok) {
      const detail = Array.isArray(data.detail)
        ? data.detail.map(d => d.loc.slice(-1)[0] + ": " + d.msg).join("; ")
        : (data.detail || "Server error");
      throw new Error(detail);
    }
    $("resultBox").style.display = "block";
    animatePrice(data.predicted_price_usd);
    $("pills").innerHTML =
      '<span class="pill">' + data.model + '</span>' +
      '<span class="pill">R² = ' + data.r2 + '</span>' +
      '<span class="pill">Accuracy ' + data.accuracy_percent + '%</span>' +
      '<span class="pill">Typical error ±$' + Math.round(data.typical_error_usd).toLocaleString() + '</span>';
    history.unshift({ carat: payload.carat.toFixed(2), cut: payload.cut, color: payload.color,
                      clarity: payload.clarity, price: data.predicted_price_usd });
    renderHistory();
  } catch (e) {
    $("placeholder").style.display = "block";
    showError("Could not get a prediction: " + e.message);
  } finally {
    $("spinner").style.display = "none";
    $("predictBtn").disabled = false;
  }
}

function pick(id) {
  const opts = $(id).options;
  $(id).selectedIndex = Math.floor(Math.random() * opts.length);
}

$("exampleBtn").addEventListener("click", () => {
  const c = (0.3 + Math.random() * 2.2);
  $("caratRange").value = c.toFixed(2);
  $("caratLabel").textContent = c.toFixed(2);
  ["cut", "color", "clarity"].forEach(pick);
  $("depth").value = (59 + Math.random() * 5).toFixed(1);
  $("table").value = (54 + Math.random() * 6).toFixed(1);
  $("auto").checked = true;
  syncAuto();
  predict();
});

$("predictBtn").addEventListener("click", predict);
syncAuto();
</script>
</body>
</html>'''


@app.get("/", response_class=HTMLResponse)
def home():
    return HTML_PAGE

print("HTML page ready ✅ (", len(HTML_PAGE), "characters )")

HTML page ready ✅ ( 10329 characters )


## Step 8: Run the server and open the frontend
This starts FastAPI in the background and shows the page right here in the notebook.

In [19]:
import threading, time, uvicorn

# If a server from an earlier run is still alive, stop it first so the
# NEW app / NEW model is the one being served (old one keeps the old model in memory)
try:
    server.should_exit = True
    time.sleep(2)
    print("Stopped previous server.")
except NameError:
    pass

PORT = 8000
config = uvicorn.Config(app, host="0.0.0.0", port=PORT, log_level="warning")
server = uvicorn.Server(config)
threading.Thread(target=server.run, daemon=True).start()
time.sleep(2)
print(f"Server running on port {PORT} ✅")

try:
    # Colab: show the web page inside the notebook
    from google.colab import output
    output.serve_kernel_port_as_iframe(PORT, height=820)
except ImportError:
    # Running locally: just open this link in your browser
    print(f"Open http://localhost:{PORT} in your browser")

Server running on port 8000 ✅


<IPython.core.display.Javascript object>

In [20]:
server.should_exit = True
print("Server stopping...")

Server stopping...
